<a href="https://colab.research.google.com/github/cd-public/compute/blob/main/ps/FA_Reg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finite Automata and Regular Languages
*Theory of Computation: Problem Set One*

## Introduction

### Deterministic Finite Automaton (DFA)

A Deterministic Finite Automaton (DFA) is a 5-tuple:

**(Q, Σ, δ, q₀, F)**

Where:

* **Q:** A finite, non-empty set of *states*.

* **Σ:** A finite, non-empty set of *input symbols* (the alphabet).

* **δ:** A *transition function*, where δ: Q × Σ → Q.  This function takes a state and an input symbol as arguments and returns a single next state.  This is the "deterministic" part - for any given state and input, there is *exactly one* next state.

* **q₀:** The *initial state*, where q₀ ∈ Q. The DFA starts processing input from this state.

* **F:** A set of *accepting states* (or final states), where F ⊆ Q.  If the DFA ends in a state that is a member of F after processing the entire input string, the string is considered accepted.

**In simpler terms:**

A DFA is a machine that reads a string of symbols one at a time.  It starts in a specific state (q₀). For each symbol it reads, it transitions to another state according to the transition function (δ).  After reading the entire string, if the DFA is in one of the accepting states (F), the string is accepted; otherwise, it is rejected.  The key feature is that the next state is *uniquely determined* by the current state and the current input symbol.

In [53]:
# We can also express a DFA in Pythonic JSON, by creating some unique Python object to represent each state.
# Inputs to this DFA would be lists of numerical zero and one, representing bit strings.

q_1, q_2, q_3 = "q_1", "q_2", "q_3"

# define M_1
Q = {q_1, q_2, q_3}
S = {0, 1}
d = {
    q_1 : { 0:q_1, 1:q_2 },
    q_2 : { 0:q_1, 1:q_3 },
    q_3 : { 0:q_3, 1:q_3 }
}
M_1 = (Q,S,d,q_1,{q_3})

### Regular Languages

Regular languages over an alphabet Σ are defined recursively as follows:

1. **Base Cases:**
    * The empty language, ∅, is a regular language.
    * The language containing the empty string, {ε}, is a regular language.
    * For each symbol 'a' in Σ, the language {a} is a regular language.

2. **Inductive Steps (Regular Operations):**
    * If L₁ and L₂ are regular languages, then:
        * **Union:** L₁ ∪ L₂ = {w | w ∈ L₁ or w ∈ L₂} is a regular language.  (The set of strings that are in either L₁ or L₂.)
        * **Concatenation:** L₁L₂ = {xy | x ∈ L₁ and y ∈ L₂} is a regular language. (The set of strings formed by concatenating a string from L₁ with a string from L₂.)
        * **Kleene Star:** L₁* = {ε} ∪ L₁ ∪ L₁L₁ ∪ L₁L₁L₁ ∪ ... is a regular language. (The set of strings formed by concatenating zero or more strings from L₁.)

We note the below we will implement *recognizers* not *generators* in deviation from theory, but we will use regular operations to create recognizers.

### Construct first the atomics

In [54]:
# We can express a Regular Language in Python as functions that return true on reading some input in Σ^n

Sigma = [0,1]

empty_lng = lambda x: x == []
empty_str = lambda x: x == [""]
# https://stackoverflow.com/questions/6076270/lambda-function-in-list-comprehensions
atomics_n = [lambda x, b=s: x == [b] for s in Sigma]

### Construct  then the Regular Operations

In [55]:
union_lng = lambda L1, L2: lambda x: L1(x) or L2(x)
concatlng = lambda L1, L2: lambda x: any([L1(x[:i]) and L2(x[i:]) for i in range(len(x) + 1)])
kleenelng = lambda L1: lambda x: empty_lng(x) or L1(x) or concatlng(L1,kleenelng(L1))(x)

### Construct an example

In [56]:
L_1 = union_lng(kleenelng(atomics_n[0]),kleenelng(atomics_n[1]))
L_1([0,0,0,0]), L_1([1,1,1,1]), L_1([0,0,1,1]), L_1([0,0]),

(True, True, False, True)

## Problem 1

Construct the regular language equivalent to the DFA shown in the introduction, using the atomics and the regular operations.

In [57]:
any_symbol = union_lng(atomics_n[0], atomics_n[1])

any_string = kleenelng(any_symbol)

two_ones = concatlng(atomics_n[1], atomics_n[1])

anything = concatlng(concatlng(any_string, two_ones), any_string)

## Problem 2

* Construct the DFA equivalent to the regular language shown in the introduction, using the states.

In [58]:
q_0, q_1, q_2, q_3 = "q_0", "q_1", "q_2", "q_3"

#Empty Strings, All 0s, All 1s, at least one of each

Q = {q_0, q_1, q_2, q_3}
S = {0, 1}
F = {q_0, q_1, q_2}
d = {
    q_0 : { 0:q_1, 1:q_2 },
    q_1 : { 0:q_1, 1:q_3 },
    q_2 : { 1:q_2, 0:q_3 },
    q_3 : { 0:q_3, 1:q_3 }
}
M_L1 = (Q,S,d,q_0,{q_0, q_1, q_2})

## Problem 3


* Create a function that, given a regular language and a "bit string", by which we mean a Pythonic list of numerical zero and one values, returns a Pythonic boolean if the bit string is recognized by the DFA.
* Test your solutions to Problem 1 and Problem 2 over no fewer than 10 bit strings, showing equivalence in all cases.

In [59]:
def simulate(M, x):
    Q, S, d, q_0, F = M
    current = q_0
    for symbol in x:
        current = d[current][symbol]

    return current in F


## Problem 4

* Create a function that, given a DFA, constructs the equivalent regular language and returns this regular language as a function.

In [60]:
def dfa_to_regex(M):
    Q, S, d, q_0, F = M
    states = list(Q)
    n = len(states)
    start = states.index(q_0)
    accept = [states.index(f) for f in F]

    empty_set = lambda x: False

    R = [[empty_set for _ in range(n)] for _ in range(n)]
    for i in range (n):
        for j in range(n):
            lang = empty_set
            for a in S:
                if d[states[i]][a] == states[j]:
                    lang = union_lng(lang, atomics_n[a])
            if i == j:
                lang = union_lng(lang, empty_lng)
            R[i][j] = lang

    for k in range(n):
        new_R = [[None]*n for _ in range(n)]
        for i in range(n):
            for j in range(n):
                via_k = concatlng(concatlng(R[i][k], kleenelng(R[k][k])), R[k][j])
                new_R[i][j] = union_lng(R[i][j], via_k)
        R = new_R
    
    result = empty_set
    for f_idx in accept:
        result = union_lng(result, R[start][f_idx])
    return result